In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys

project_pth=os.path.join(os.getcwd(),'..','..')

sys.path.append(project_pth)

from utils.transformations import reusable

In [0]:
df_user = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/DimUser/schema")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .load("abfss://bronze@sidazureproject.dfs.core.windows.net/DimUser"))





In [0]:
df_user=df_user.withColumn("user_name",upper(col("user_name")))


In [0]:
class reusable:
    def dropColumns(self, df, columns):
        return df.drop(*columns)



In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])


In [0]:
df_user=df_user.dropDuplicates(['user_id'])

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@sidazureproject.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.DimUser")



In [0]:
df_art = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/DimArt/schema")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .load("abfss://bronze@sidazureproject.dfs.core.windows.net/DimArtist"))

In [0]:
df_art_obj= reusable()

df_art=df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art=df_art.dropDuplicates(['artist_id'])

In [0]:
df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/DimArt/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@sidazureproject.dfs.core.windows.net/DimArt/data")\
    .toTable("spotify_cata.silver.DimArtist")

In [0]:
df_trk = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/DimTrack/schema")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .load("abfss://bronze@sidazureproject.dfs.core.windows.net/DimTrack"))

In [0]:
df_trk=df_trk.withColumn("durFlag",when(col("duration_sec")<150,"low")\
                                   .when(col("duration_sec")<300,"Medium")\
                                       .otherwise("high"))
                                

In [0]:
df_trk=df_trk.withColumn("track_name",regexp_replace(col("track_name"),'-',' '))

In [0]:
df_trk_obj= reusable()

df_trk=df_trk_obj.dropColumns(df_trk,['_rescued_data'])


In [0]:
df_trk.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@sidazureproject.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.DimTrack")

In [0]:
df_date = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/DimDate/schema")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .load("abfss://bronze@sidazureproject.dfs.core.windows.net/DimDate"))

In [0]:
df_date_obj= reusable()

df_date=df_date_obj.dropColumns(df_date,['_rescued_data'])

In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@sidazureproject.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.DimDate")

In [0]:
df_fact = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/FactStream/schema")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .load("abfss://bronze@sidazureproject.dfs.core.windows.net/FactStream"))

In [0]:
df_fact_obj= reusable()

df_fact=df_fact_obj.dropColumns(df_fact,['_rescued_data'])

In [0]:
df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation",
        "abfss://silver@sidazureproject.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@sidazureproject.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.FactStream")

In [0]:
%sql
select * from spotify_cata.gold.dimtrack where track_id IN (46,5)